In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

GOLD_PATH = "output/gold"

In [2]:
fact = spark.read.csv(f"{GOLD_PATH}/fact_order_items", header=True, inferSchema=True)
dim_products = spark.read.csv(f"{GOLD_PATH}/dim_products", header=True, inferSchema=True)
dim_sellers = spark.read.csv(f"{GOLD_PATH}/dim_sellers", header=True, inferSchema=True)

In [3]:
fact.createOrReplaceTempView("fact_order_items")
dim_products.createOrReplaceTempView("dim_products")
dim_sellers.createOrReplaceTempView("dim_sellers")

In [4]:
query_1 = """
WITH monthly_revenue AS (
    SELECT
        p.product_category_name,
        YEAR(f.order_date) AS year,
        MONTH(f.order_date) AS month,
        SUM(f.price + f.freight_value) AS monthly_revenue,
        COUNT(*) AS txn_count
    FROM fact_order_items f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_category_name, YEAR(f.order_date), MONTH(f.order_date)
    HAVING COUNT(*) >= 10
),

ranked AS (
    SELECT *,
        RANK() OVER (
            PARTITION BY year, month
            ORDER BY monthly_revenue DESC
        ) AS monthly_rank
    FROM monthly_revenue
),

top_categories AS (
    SELECT product_category_name
    FROM monthly_revenue
    GROUP BY product_category_name
    ORDER BY SUM(monthly_revenue) DESC
    LIMIT 5
),

final_calc AS (
    SELECT
        r.product_category_name,
        r.year,
        r.month,
        r.monthly_revenue,
        r.monthly_rank,
        LAG(r.monthly_revenue) OVER (
            PARTITION BY r.product_category_name
            ORDER BY r.year, r.month
        ) AS prev_revenue,
        AVG(r.monthly_revenue) OVER (
            PARTITION BY r.product_category_name
            ORDER BY r.year, r.month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rolling_3m_avg_revenue
    FROM ranked r
    WHERE r.product_category_name IN (SELECT product_category_name FROM top_categories)
)

SELECT
    product_category_name,
    year,
    month,
    monthly_revenue,
    monthly_rank,
    CASE 
        WHEN prev_revenue IS NULL THEN NULL
        ELSE (monthly_revenue - prev_revenue) / prev_revenue
    END AS mom_growth_pct,
    rolling_3m_avg_revenue
FROM final_calc
ORDER BY product_category_name, year, month
"""

In [5]:
result_q1 = spark.sql(query_1)
result_q1.show(20, False)

+---------------------+----+-----+-----------------+------------+--------------------+----------------------+
|product_category_name|year|month|monthly_revenue  |monthly_rank|mom_growth_pct      |rolling_3m_avg_revenue|
+---------------------+----+-----+-----------------+------------+--------------------+----------------------+
|beleza_saude         |2018|6    |1351.0           |4           |NULL                |1351.0                |
|beleza_saude         |2018|7    |122743.9800000001|1           |89.85416728349378   |62047.49000000005     |
|beleza_saude         |2018|8    |137206.0700000001|1           |0.11782321218523291 |87100.35000000006     |
|cama_mesa_banho      |2018|6    |1538.54          |3           |NULL                |1538.54               |
|cama_mesa_banho      |2018|7    |68708.38         |4           |43.65816943335891   |35123.46              |
|cama_mesa_banho      |2018|8    |75207.83000000002|3           |0.09459472046932282 |48484.916666666664    |
|esporte_l

In [6]:
query_2 = """
WITH seller_metrics AS (
    SELECT
        f.seller_key,
        s.seller_id,
        s.seller_state,
        COUNT(*) AS total_orders,
        SUM(f.price + f.freight_value) AS total_revenue,
        AVG(f.days_delivery_vs_estimate) AS avg_days_vs_estimate,
        SUM(CASE WHEN f.is_late_delivery = true THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS late_delivery_rate
    FROM fact_order_items f
    JOIN dim_sellers s
        ON f.seller_key = s.seller_key
    GROUP BY f.seller_key, s.seller_id, s.seller_state
    HAVING COUNT(*) >= 20
),

percentiles AS (
    SELECT *,
        PERCENT_RANK() OVER (ORDER BY total_revenue) AS revenue_pctl,
        1 - PERCENT_RANK() OVER (ORDER BY late_delivery_rate) AS on_time_pctl,
        1 - PERCENT_RANK() OVER (ORDER BY avg_days_vs_estimate) AS speed_pctl
    FROM seller_metrics
),

final_scores AS (
    SELECT *,
        (0.4 * on_time_pctl + 0.3 * speed_pctl + 0.3 * revenue_pctl) AS composite_score
    FROM percentiles
)

SELECT
    seller_id,
    seller_state,
    total_orders,
    total_revenue,
    late_delivery_rate,
    avg_days_vs_estimate,
    on_time_pctl,
    speed_pctl,
    revenue_pctl,
    composite_score,
    RANK() OVER (ORDER BY composite_score DESC) AS overall_rank
FROM final_scores
ORDER BY overall_rank
"""

In [7]:
result_q2 = spark.sql(query_2)
result_q2.show(20, False)

+--------------------------------+------------+------------+------------------+------------------+--------------------+------------------+------------------+-------------------+------------------+------------+
|seller_id                       |seller_state|total_orders|total_revenue     |late_delivery_rate|avg_days_vs_estimate|on_time_pctl      |speed_pctl        |revenue_pctl       |composite_score   |overall_rank|
+--------------------------------+------------+------------+------------------+------------------+--------------------+------------------+------------------+-------------------+------------------+------------+
|3d871de0142ce09b7081e2b9d1733cb1|SP          |59          |7108.76           |0.0000000000000000|-14.728813559322035 |1.0               |0.9612903225806452|0.7548387096774194 |0.9148387096774194|1           |
|58f1a6197ed863543e0136bdedb3fce2|MG          |36          |6382.240000000001 |0.0000000000000000|-15.294117647058824 |1.0               |0.9806451612903225|0.7